# 🚀 md-editor 端侧小模型: Google Colab T4 一键训练与发布

本 Notebook 可在 **Google Colab (免费 T4 GPU)** 上一键完成：
1. **环境自检**：检测 NVIDIA Tesla T4 GPU 与 CUDA 环境
2. **多任务微调**：运行 RFC-002 SFT 微调（0.5B 约 15 分钟，1.5B 约 35 分钟）
3. **自动量化**：通过 `llama.cpp` 转换为 `Q4_K_M` GGUF 格式
4. **自动发布 / 下载**：自动发布至 GitHub Releases，若无 Token 则提供一键本地下载

> 🔄 **自动同步**：每次运行都会自动从 GitHub 拉取最新修复代码，无需手动重克隆！

In [ ]:
#@title ⚙️ [1/4] 配置训练参数与基座选择
#@markdown 请在右侧面板选择你要训练的模型规格：

model_tier = "0.5B (Lite - 约15分钟)" #@param ["0.5B (Lite - 约15分钟)", "1.5B (Standard - 约35分钟)"]
version_tag = "v1.0.0" #@param {type:"string"}

if "1.5B" in model_tier:
    BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
else:
    BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"🎯 已选择基座: {BASE_MODEL}")
print(f"🏷️ 版本标签:   {version_tag}")

# 验证 GPU 状态
!nvidia-smi

In [ ]:
#@title 📦 [2/4] 克隆/更新仓库并安装依赖环境
import os
if not os.path.exists('/content/md-editor-models'):
    !git clone https://github.com/wmasfoe/md-editor-models.git /content/md-editor-models
else:
    !git -C /content/md-editor-models pull origin master

%cd /content/md-editor-models
!git pull origin master
!pip install trl peft pangu datasets transformers accelerate sentencepiece

In [ ]:
#@title 🔑 [3/4] 配置 GitHub Token (可选，用于自动发布 Release)
#@markdown 若不填写，训练完成后可直接点击下方的下载按钮将 GGUF 文件保存到电脑：

import os
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None

if not token:
    token = input("请输入你的 GitHub Token (直接回车可跳过，跳过则不自动发布 Release): ").strip()

if token:
    os.environ['GH_TOKEN'] = token
    os.environ['GITHUB_TOKEN'] = token
    print("✅ GitHub Token 配置成功！")
else:
    print("ℹ️ 未提供 Token，训练完成后将支持直接从浏览器下载 GGUF 文件。")

In [ ]:
#@title 🚀 [4/4] 启动一键训练、量化与产物交付！
# 确保始终运行最新代码
!git pull origin master
!chmod +x scripts/release_model.sh
!./scripts/release_model.sh $version_tag $BASE_MODEL

# 如果没有配置 GitHub Token，提供浏览器直接下载弹窗
if not os.environ.get('GH_TOKEN'):
    from google.colab import files
    import glob
    gguf_files = glob.glob("output/*-Q4_K_M.gguf")
    if gguf_files:
        print("\n📥 正在拉起浏览器下载 GGUF 模型与 Manifest...")
        files.download(gguf_files[0])
        if os.path.exists("output/manifest.json"):
            files.download("output/manifest.json")